In [ ]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
import random
import time
import html
import re

In [ ]:
def get_cdx_urls(year_from, year_to):

    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1'
    }

    url = f'http://web.archive.org/cdx/search/cdx?url=tunisie-annonce.com&from={year_from}&to={year_to}&filter=statuscode:200&output=json'
    try:
        response = requests.get(url, timeout=30, headers = headers)
        response.raise_for_status()
        data = json.loads(response.text)

        wayback_urls = []
        for row in data [1:]: #skip header
            timestamp = row[1]
            original_url = row[2]

            if not original_url.endswith('AnnoncesAuto.asp'):
                if original_url.endswith('/'):
                    original_url += 'AnnoncesAuto.asp'
                else:
                    original_url += '/AnnoncesAuto.asp'
                
            wayback_url = f'https://web.archive.org/web/{timestamp}/{original_url}'
            wayback_urls.append(wayback_url)
        
        return wayback_urls
    except Exception as e:
        print(f"Error getting CDX URLs: {e}")
        return []

In [ ]:
def scrape_link_details(soup, page):
    
    all_ads = []
    try:
        large_table = soup.find('table', width='767', bgcolor=re.compile('white', re.IGNORECASE))
        if not large_table:
            print(f'✗ Error scraping page: {page}- No large table found')
            return None
        
        main_table = large_table

        #removing the vertical menu table
        vertical_menu_table = main_table.find('table', class_='MenuVertical')
        if vertical_menu_table:
            vertical_menu_table.decompose()
        
        ads = main_table.find_all('tr', class_='Tableau1')
        
        for ad in ads:
            
            ad_details = ad.find_all('td')[1::2] #accessing odd indices (ignoring images)
            if not ad_details or len(ad_details)<7:
                print(f'✗ Error scraping page: {page}- Ad details not found')
                continue #we go to the next iteration
            
            info={}
            info['location'] = ad_details[0].find('a').get_text(strip=True) if ad_details[0].find('a') else ''
            info['brand'] = ad_details[1].get_text(strip=True) if ad_details[1].get_text(strip=True) else ''
            info['model'] = ad_details[2].get_text(strip=True) if ad_details[2].get_text(strip=True) else ''

            raw_text = ad_details[3].find('a')['onmouseover'].strip() if ad_details[3].find('a') and ad_details[3].find('a').get('onmouseover') else ''
            info['description']=''
            if raw_text:
                cleaned = raw_text.replace("return escape('", "").replace("');", "")
                info['description'] = BeautifulSoup(html.unescape(cleaned), 'html.parser').get_text()

            info['circulation-date'] = ad_details[4].get_text(strip=True) if ad_details[4].get_text(strip=True) else ''
            info['price'] = ad_details[5].get_text(strip=True) if ad_details[5].get_text(strip=True) else ''
            info['publish-date'] = ad_details[6].get_text(strip=True) if ad_details[6].get_text(strip=True) else ''

            wanted_keywords = ['cherche', 'recherche']
            description_lower = info['description'].lower()
            if any(keyword in description_lower for keyword in wanted_keywords):
                continue

            all_ads.append(info)
        
    except Exception as e:
        print(f'✗ Error extracting ads from: {page}: {e}')

        with open('debug_page.html', 'w', encoding='utf-8') as f:
            f.write(str(soup))
        print("Saved HTML content to debug_page.html for inspection")

    return all_ads

In [ ]:
def scrape_urls(start, end):

    df = pd.DataFrame(columns=[
            'location','brand','model','circulation-date','price','publish-date','description','link'
    ])
    successful_scrapes = 0
    failed_scrapes = 0
    page = 1

    #add beginning and ending years
    urls = get_cdx_urls(start, end)
    print(f'We have {len(urls)} url to scrape')

    for link in urls:
        print(f'\n{"="*50}')
        print(f'Scraping link number: {page}')
        print(f'URL: {link}')
        print(f'{"="*50}')

        try:
            headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1'}
            response = requests.get(link, timeout=30, headers=headers)
            response.raise_for_status() #generates an error if the code is not 200

            print(f"Response Status: {response.status_code}")

            soup = BeautifulSoup(response.text, 'lxml')
            ads_data = scrape_link_details(soup, page)

            if ads_data:
                    new_df = pd.DataFrame(ads_data)
                    new_df['link'] = link
                    df = pd.concat([df, new_df], ignore_index = True)
                    successful_scrapes += 1
                    print(f'✓ Successfully scraped link number: {page}')
            else:
                    print(f'✗ No extracted data from link number: {page}')

            
        except requests.exceptions.RequestException as e:
            print(f'✗ Failed to scrape link number: {page}: {e}')
            failed_scrapes += 1

        except Exception as e:
            print(f'✗ Unexpected error occured scrapig link number: {page}: {e}')
            failed_scrapes += 1
        
        page += 1
        time.sleep(random.uniform(7,15))

    total_processed = failed_scrapes + successful_scrapes 
    success_rate = (successful_scrapes / len(urls)) * 100 if urls else 0
    print(f'\n{"="*50}')
    print(f'SCRAPING COMPLETE')
    print(f'{"="*50}')
    print(f'Total URLs processed: {total_processed}')
    print(f'Successful scrapes: {successful_scrapes}')
    print(f'Failed scrapes: {failed_scrapes}')
    print(f'Success rate: {success_rate:.1f}%')
    print(f'Total ads collected: {len(df)}')

    return df

In [ ]:
def save_results(df, filename):
    try:
        df.to_csv(filename, index=False)
    except Exception as e:
        print(f'Could not save: {e}')

In [ ]:
#USE THE FULL CODE:
'''Modify start and end dates'''
start = 2007
end = 2025
df = scrape_urls(start, end)
print('Dataframe filled')
print("Samle of the Dataframe")
print(df.head())
print(f'Dataframe shape: {df.shape}')

'''Modify the file name'''
save_results(df, 'wayback_data.csv')